# Notebook 02 — Data Cleaning & Standardization

## Objective
This notebook transforms the raw Los Angeles traffic collision dataset into a clean, standardized, and analysis-ready base table.

## Why this notebook matters
Based on the project roadmap, this phase is responsible for:
- standardizing column names
- parsing and validating date fields
- cleaning and structuring time values
- deriving core temporal features
- reviewing invalid coordinates
- standardizing selected categorical text fields
- preparing clean base outputs for downstream modeling and analytics

## Analytical role in the project
This notebook is the bridge between:
1. **Notebook 01 — Data Audit & Profiling**
2. **Notebook 03 — MO Codes Engineering**
3. downstream outputs for **SQL, Excel, Tableau, and Power BI**

## Main expected outputs
- `collisions_clean.csv`
- `dim_date.csv`
- `dim_area.csv`
- `dim_premise.csv`
- `cleaning_summary_report.csv`

## Methodological note
The cleaning logic in this notebook follows a conservative analytical approach:
- preserve raw information whenever possible
- standardize structure before deriving insights
- avoid unjustified imputation
- make data quality issues visible through validation checks and flags
- keep Python as the analytical source of truth for downstream tools

In [1]:
# ============================================================
# Notebook 02 — Data Cleaning & Standardization
# Cell 1: Environment setup, imports, and project paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Display settings for better inspection
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

# ------------------------------------------------------------
# Resolve project root dynamically
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

# Create target folders if they do not already exist
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Define input file paths
# ------------------------------------------------------------
COLLISIONS_FILE = RAW_DIR / "Traffic_Collision_Data_from_2010_to_Present_20260127.csv"
MO_CODES_FILE = RAW_DIR / "Mo Codes.xlsx"
MO_CODES_CLASSIFIED_FILE = RAW_DIR / "MO_Codes_Classified_for_Traffic_Project.csv"

print("=" * 70)
print("Notebook 02 — Environment Setup")
print("=" * 70)
print(f"Current directory : {CURRENT_DIR}")
print(f"Project root      : {PROJECT_ROOT}")
print(f"Raw data dir      : {RAW_DIR}")
print(f"Cleaned data dir  : {CLEANED_DIR}")
print(f"Tables dir        : {TABLES_DIR}")
print(f"Figures dir       : {FIGURES_DIR}")
print("-" * 70)
print("Expected input files:")
print(f"Collision file            -> {COLLISIONS_FILE}")
print(f"MO codes reference file   -> {MO_CODES_FILE}")
print(f"MO classified file        -> {MO_CODES_CLASSIFIED_FILE}")
print("-" * 70)
print("Existence check:")
print(f"Collision file exists?          {COLLISIONS_FILE.exists()}")
print(f"MO codes reference exists?      {MO_CODES_FILE.exists()}")
print(f"MO classified file exists?      {MO_CODES_CLASSIFIED_FILE.exists()}")
print("=" * 70)

Notebook 02 — Environment Setup
Current directory : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\notebooks
Project root      : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project
Raw data dir      : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\raw
Cleaned data dir  : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned
Tables dir        : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables
Figures dir       : c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\figures
----------------------------------------------------------------------
Expected input files:
Collision file            -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\raw\Traffic_Collision_Data_from_2010_to_Present_20260127.csv
MO codes reference file   -> c:\Users\MKamal\Desktop\Project\with

## Step 1 — Load Raw Collision Data and Establish the Cleaning Baseline

In this step, we load the raw collision dataset and document its starting structure before any transformation.

### Why this step is important
A scientific cleaning workflow should begin with a clearly documented baseline.  
Before renaming columns, parsing dates, or standardizing fields, we must confirm:

- dataset shape
- column names
- column order
- first-level structural consistency

### Analytical purpose
This gives us a controlled starting point for all downstream cleaning decisions and allows us to compare:
- raw input state
- cleaned output state
- structural changes introduced intentionally during the notebook

### Method note
At this stage, we do **not** modify the dataset.  
We only load, inspect, and document the raw input as the starting reference for Notebook 02.

In [2]:
# ============================================================
# Cell 2: Load raw collision data and inspect the baseline
# ============================================================

# Load the raw collision dataset
df_raw = pd.read_csv(COLLISIONS_FILE, low_memory=False)

# Create a working copy for later cleaning steps
df = df_raw.copy()

print("=" * 70)
print("Raw Collision Dataset Loaded Successfully")
print("=" * 70)
print(f"Raw shape          : {df_raw.shape}")
print(f"Working copy shape : {df.shape}")
print(f"Number of columns  : {len(df.columns)}")
print("-" * 70)

print("Column names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("-" * 70)
print("First 5 rows preview:")
display(df.head())

print("-" * 70)
print("DataFrame info:")
print(df.info())
print("=" * 70)

Raw Collision Dataset Loaded Successfully
Raw shape          : (621677, 18)
Working copy shape : (621677, 18)
Number of columns  : 18
----------------------------------------------------------------------
Column names:
01. DR Number
02. Date Reported
03. Date Occurred
04. Time Occurred
05. Area ID
06. Area Name
07. Reporting District
08. Crime Code
09. Crime Code Description
10. MO Codes
11. Victim Age
12. Victim Sex
13. Victim Descent
14. Premise Code
15. Premise Description
16. Address
17. Cross Street
18. Location
----------------------------------------------------------------------
First 5 rows preview:


,DR Number,Date Reported,Date Occurred,Time Occurred,Area ID,Area Name,Reporting District,Crime Code,Crime Code Description,MO Codes,Victim Age,Victim Sex,Victim Descent,Premise Code,Premise Description,Address,Cross Street,Location
0,212013850,09/03/2021,09/02/2021,2335,20,Olympic,2021,997,TRAFFIC COLLISION,3004 3027 3034 4027 3036 3101 3401 3701,25.0,F,W,101.0,STREET,WILTON PL,6TH ST,"(34.063, -118.3141)"
1,221417787,10/17/2022,10/17/2022,1620,14,Pacific,1406,997,TRAFFIC COLLISION,4027 3011 3028 3034 3037 3101 3401 3701,21.0,NaN,NaN,101.0,STREET,NATIONAL BL,MOTOR AV,"(34.029, -118.4113)"
2,221418141,10/26/2022,10/26/2022,1135,14,Pacific,1434,997,TRAFFIC COLLISION,4027 3011 3025 3034 3037 3101 3401 3701,21.0,NaN,NaN,101.0,STREET,PALMS BL,ROSEWOOD AV,"(34.0052, -118.4478)"
3,222017859,12/01/2022,12/01/2022,230,20,Olympic,2044,997,TRAFFIC COLLISION,3003 0913 3026 3035 3037 3101 3401 3701 4020,33.0,M,H,101.0,STREET,IROLO ST,SAN MARINO ST,"(34.0545, -118.3009)"
4,190319651,08/24/2019,08/24/2019,450,3,Southwest,356,997,TRAFFIC COLLISION,3036 3004 3026 3101 4003,22.0,M,H,101.0,STREET,JEFFERSON BL,NORMANDIE AV,"(34.0255, -118.3002)"


----------------------------------------------------------------------
DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 621677 entries, 0 to 621676
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   DR Number               621677 non-null  int64  
 1   Date Reported           621677 non-null  str    
 2   Date Occurred           621677 non-null  str    
 3   Time Occurred           621677 non-null  int64  
 4   Area ID                 621677 non-null  int64  
 5   Area Name               621677 non-null  str    
 6   Reporting District      621677 non-null  int64  
 7   Crime Code              621677 non-null  int64  
 8   Crime Code Description  621677 non-null  str    
 9   MO Codes                534353 non-null  str    
 10  Victim Age              533483 non-null  float64
 11  Victim Sex              610980 non-null  str    
 12  Victim Descent          610029 non-null  str    
 13

## Step 2 — Standardize Column Names

Before performing any cleaning transformations, we standardize the raw source column names into a consistent analytical naming convention.

### Why this step matters
Raw source column names are human-readable, but they are not ideal for a reproducible analytical workflow.  
Standardized names improve:

- readability
- code consistency
- compatibility with SQL and BI tools
- downstream transformation reliability

### Naming convention used
We convert column names to a clean snake_case structure:
- lowercase only
- spaces replaced with underscores
- no special formatting ambiguity

### Methodological note
This step changes only the **column labels**, not the underlying data values.
It is therefore considered a structural standardization step, not a content-cleaning step.

In [3]:
# ============================================================
# Cell 3: Standardize column names
# ============================================================

# Preserve original column names for documentation
original_columns = df.columns.tolist()

# Define an explicit rename map for transparency and control
rename_map = {
    "DR Number": "dr_number",
    "Date Reported": "date_reported",
    "Date Occurred": "date_occurred",
    "Time Occurred": "time_occurred",
    "Area ID": "area_id",
    "Area Name": "area_name",
    "Reporting District": "reporting_district",
    "Crime Code": "crime_code",
    "Crime Code Description": "crime_code_description",
    "MO Codes": "mo_codes",
    "Victim Age": "victim_age",
    "Victim Sex": "victim_sex",
    "Victim Descent": "victim_descent",
    "Premise Code": "premise_code",
    "Premise Description": "premise_description",
    "Address": "address",
    "Cross Street": "cross_street",
    "Location": "location"
}

# Apply renaming
df = df.rename(columns=rename_map)

# Capture renamed columns
renamed_columns = df.columns.tolist()

# Build documentation table
rename_audit = pd.DataFrame({
    "original_column": original_columns,
    "standardized_column": renamed_columns
})

print("=" * 70)
print("Column Renaming Completed")
print("=" * 70)
print(f"Number of columns before renaming : {len(original_columns)}")
print(f"Number of columns after renaming  : {len(renamed_columns)}")
print("-" * 70)
print("Renaming audit table:")
display(rename_audit)

print("-" * 70)
print("Updated column names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("=" * 70)

Column Renaming Completed
Number of columns before renaming : 18
Number of columns after renaming  : 18
----------------------------------------------------------------------
Renaming audit table:


,original_column,standardized_column
0,DR Number,dr_number
1,Date Reported,date_reported
2,Date Occurred,date_occurred
3,Time Occurred,time_occurred
4,Area ID,area_id
5,Area Name,area_name
6,Reporting District,reporting_district
7,Crime Code,crime_code
8,Crime Code Description,crime_code_description
9,MO Codes,mo_codes


----------------------------------------------------------------------
Updated column names:
01. dr_number
02. date_reported
03. date_occurred
04. time_occurred
05. area_id
06. area_name
07. reporting_district
08. crime_code
09. crime_code_description
10. mo_codes
11. victim_age
12. victim_sex
13. victim_descent
14. premise_code
15. premise_description
16. address
17. cross_street
18. location


## Step 3 — Parse and Validate Date Fields

The raw date columns are currently stored as text values.  
Before deriving any temporal features, we must convert them into proper datetime fields.

### Date fields in scope
- `date_reported`
- `date_occurred`

### Why this step matters
Reliable date parsing is essential because later analysis will depend on:
- yearly trends
- monthly patterns
- weekday patterns
- reporting lag checks
- date dimension generation

### Validation logic
This step follows a controlled conversion approach:
- parse the raw text values into datetime
- count invalid conversions explicitly
- preserve the original row count
- document the success of conversion before creating derived time features

### Methodological note
At this stage, we only convert and validate the two date columns.  
Derived date attributes such as year, month, and weekday will be created only after confirming that parsing succeeded.

In [4]:
# ============================================================
# Cell 4: Parse and validate date fields
# ============================================================

# Keep a copy of raw text date columns for validation/reference
df["date_reported_raw"] = df["date_reported"]
df["date_occurred_raw"] = df["date_occurred"]

# Convert to datetime
df["date_reported"] = pd.to_datetime(df["date_reported"], errors="coerce")
df["date_occurred"] = pd.to_datetime(df["date_occurred"], errors="coerce")

# Validation summary
date_validation_summary = pd.DataFrame({
    "column": ["date_reported", "date_occurred"],
    "raw_non_null_count": [
        df["date_reported_raw"].notna().sum(),
        df["date_occurred_raw"].notna().sum()
    ],
    "parsed_non_null_count": [
        df["date_reported"].notna().sum(),
        df["date_occurred"].notna().sum()
    ],
    "failed_parse_count": [
        df["date_reported"].isna().sum(),
        df["date_occurred"].isna().sum()
    ],
    "parsed_dtype": [
        str(df["date_reported"].dtype),
        str(df["date_occurred"].dtype)
    ],
    "min_date": [
        df["date_reported"].min(),
        df["date_occurred"].min()
    ],
    "max_date": [
        df["date_reported"].max(),
        df["date_occurred"].max()
    ]
})

print("=" * 70)
print("Date Parsing Validation")
print("=" * 70)
display(date_validation_summary)

print("-" * 70)
print("Sample rows after parsing:")
display(
    df[
        [
            "date_reported_raw", "date_reported",
            "date_occurred_raw", "date_occurred"
        ]
    ].head(10)
)

print("-" * 70)
print("Current dtypes of parsed date columns:")
print(df[["date_reported", "date_occurred"]].dtypes)
print("=" * 70)

Date Parsing Validation


,column,raw_non_null_count,parsed_non_null_count,failed_parse_count,parsed_dtype,min_date,max_date
0,date_reported,621677,621677,0,datetime64[us],2010-01-01,2025-03-08
1,date_occurred,621677,621677,0,datetime64[us],2010-01-01,2025-03-08


----------------------------------------------------------------------
Sample rows after parsing:


,date_reported_raw,date_reported,date_occurred_raw,date_occurred
0,09/03/2021,2021-09-03,09/02/2021,2021-09-02
1,10/17/2022,2022-10-17,10/17/2022,2022-10-17
2,10/26/2022,2022-10-26,10/26/2022,2022-10-26
3,12/01/2022,2022-12-01,12/01/2022,2022-12-01
4,08/24/2019,2019-08-24,08/24/2019,2019-08-24
5,08/30/2019,2019-08-30,08/30/2019,2019-08-30
6,08/25/2019,2019-08-25,08/25/2019,2019-08-25
7,11/20/2019,2019-11-20,11/20/2019,2019-11-20
8,08/30/2019,2019-08-30,08/30/2019,2019-08-30
9,07/06/2019,2019-07-06,07/06/2019,2019-07-06


----------------------------------------------------------------------
Current dtypes of parsed date columns:
date_reported    datetime64[us]
date_occurred    datetime64[us]
dtype: object


## Step 4 — Derive Core Temporal Features from the Occurrence Date

After confirming successful date parsing, we now derive the main temporal attributes needed for analysis.

### Reference date used
The primary analytical date in this project is:

- `date_occurred`

This reflects when the collision actually happened, which makes it the correct basis for:
- yearly collision trends
- monthly seasonality
- weekday distribution
- time-based dashboard slicing
- date dimension construction

### Features created in this step
From `date_occurred`, we derive:

- `occ_year`
- `occ_month`
- `occ_month_name`
- `occ_weekday`
- `occ_weekday_name`

### Methodological note
These features are derived only after validating that the source date field was parsed successfully.
This ensures that downstream time analysis is based on structured datetime values rather than raw strings.

In [5]:
# ============================================================
# Cell 5: Derive core temporal features from date_occurred
# ============================================================

# Derive temporal fields from occurrence date
df["occ_year"] = df["date_occurred"].dt.year
df["occ_month"] = df["date_occurred"].dt.month
df["occ_month_name"] = df["date_occurred"].dt.month_name()
df["occ_weekday"] = df["date_occurred"].dt.dayofweek   # Monday=0, Sunday=6
df["occ_weekday_name"] = df["date_occurred"].dt.day_name()

# Build a compact validation summary
temporal_feature_summary = pd.DataFrame({
    "feature": [
        "occ_year",
        "occ_month",
        "occ_month_name",
        "occ_weekday",
        "occ_weekday_name"
    ],
    "non_null_count": [
        df["occ_year"].notna().sum(),
        df["occ_month"].notna().sum(),
        df["occ_month_name"].notna().sum(),
        df["occ_weekday"].notna().sum(),
        df["occ_weekday_name"].notna().sum()
    ],
    "unique_values": [
        df["occ_year"].nunique(dropna=True),
        df["occ_month"].nunique(dropna=True),
        df["occ_month_name"].nunique(dropna=True),
        df["occ_weekday"].nunique(dropna=True),
        df["occ_weekday_name"].nunique(dropna=True)
    ]
})

print("=" * 70)
print("Temporal Feature Derivation Completed")
print("=" * 70)
display(temporal_feature_summary)

print("-" * 70)
print("Sample preview of derived temporal fields:")
display(
    df[
        [
            "date_occurred",
            "occ_year",
            "occ_month",
            "occ_month_name",
            "occ_weekday",
            "occ_weekday_name"
        ]
    ].head(10)
)

print("-" * 70)
print("Distinct year values:")
print(sorted(df["occ_year"].dropna().unique().tolist())[:5], "...", sorted(df["occ_year"].dropna().unique().tolist())[-5:])

print("-" * 70)
print("Distinct month values:")
print(sorted(df["occ_month"].dropna().unique().tolist()))

print("-" * 70)
print("Distinct weekday values:")
print(sorted(df["occ_weekday"].dropna().unique().tolist()))
print(df["occ_weekday_name"].dropna().unique().tolist())

print("=" * 70)

Temporal Feature Derivation Completed


,feature,non_null_count,unique_values
0,occ_year,621677,16
1,occ_month,621677,12
2,occ_month_name,621677,12
3,occ_weekday,621677,7
4,occ_weekday_name,621677,7


----------------------------------------------------------------------
Sample preview of derived temporal fields:


,date_occurred,occ_year,occ_month,occ_month_name,occ_weekday,occ_weekday_name
0,2021-09-02,2021,9,September,3,Thursday
1,2022-10-17,2022,10,October,0,Monday
2,2022-10-26,2022,10,October,2,Wednesday
3,2022-12-01,2022,12,December,3,Thursday
4,2019-08-24,2019,8,August,5,Saturday
5,2019-08-30,2019,8,August,4,Friday
6,2019-08-25,2019,8,August,6,Sunday
7,2019-11-20,2019,11,November,2,Wednesday
8,2019-08-30,2019,8,August,4,Friday
9,2019-07-06,2019,7,July,5,Saturday


----------------------------------------------------------------------
Distinct year values:
[2010, 2011, 2012, 2013, 2014] ... [2021, 2022, 2023, 2024, 2025]
----------------------------------------------------------------------
Distinct month values:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
----------------------------------------------------------------------
Distinct weekday values:
[0, 1, 2, 3, 4, 5, 6]
['Thursday', 'Monday', 'Wednesday', 'Saturday', 'Friday', 'Sunday', 'Tuesday']


## Step 5 — Clean and Structure the Collision Time Field

The raw `time_occurred` field is currently stored as an integer-like HHMM value.  
To make it analytically usable, we need to standardize and validate its structure.

### Why this step matters
Time is essential for traffic collision analysis because it supports:
- hourly collision patterns
- peak-period analysis
- daypart segmentation
- time-based filtering in dashboards

### Transformation logic
In this step, we:
- convert the raw numeric value into a zero-padded 4-digit string
- split it into hour and minute components
- validate whether the extracted hour and minute are logically valid
- create a structured time interpretation without altering the original raw value

### Methodological note
This is a controlled transformation step.  
We preserve the original `time_occurred` field and derive structured fields from it for downstream analysis.

In [6]:
# ============================================================
# Cell 6: Clean and structure time_occurred
# ============================================================

# Preserve original raw time field
df["time_occurred_raw"] = df["time_occurred"]

# Convert to string and zero-pad to 4 digits
df["time_occurred_str"] = df["time_occurred"].astype("Int64").astype(str).str.zfill(4)

# Extract hour and minute components
df["occ_hour"] = pd.to_numeric(df["time_occurred_str"].str[:2], errors="coerce")
df["occ_minute"] = pd.to_numeric(df["time_occurred_str"].str[2:], errors="coerce")

# Validate time logic
df["is_valid_occ_hour"] = df["occ_hour"].between(0, 23, inclusive="both")
df["is_valid_occ_minute"] = df["occ_minute"].between(0, 59, inclusive="both")
df["is_valid_occ_time"] = df["is_valid_occ_hour"] & df["is_valid_occ_minute"]

# Summary table
time_validation_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "non_null_raw_time",
        "valid_hour_count",
        "valid_minute_count",
        "fully_valid_time_count",
        "invalid_time_count",
        "distinct_hour_values",
        "distinct_minute_values"
    ],
    "value": [
        len(df),
        df["time_occurred_raw"].notna().sum(),
        df["is_valid_occ_hour"].sum(),
        df["is_valid_occ_minute"].sum(),
        df["is_valid_occ_time"].sum(),
        (~df["is_valid_occ_time"]).sum(),
        df.loc[df["is_valid_occ_hour"], "occ_hour"].nunique(dropna=True),
        df.loc[df["is_valid_occ_minute"], "occ_minute"].nunique(dropna=True)
    ]
})

print("=" * 70)
print("Time Field Structuring and Validation")
print("=" * 70)
display(time_validation_summary)

print("-" * 70)
print("Sample preview:")
display(
    df[
        [
            "time_occurred_raw",
            "time_occurred_str",
            "occ_hour",
            "occ_minute",
            "is_valid_occ_time"
        ]
    ].head(15)
)

print("-" * 70)
print("Invalid time sample (if any):")
invalid_time_sample = df.loc[
    ~df["is_valid_occ_time"],
    ["time_occurred_raw", "time_occurred_str", "occ_hour", "occ_minute"]
].head(10)

display(invalid_time_sample)

print("-" * 70)
print("Distinct valid hours:")
print(sorted(df.loc[df["is_valid_occ_hour"], "occ_hour"].dropna().unique().tolist()))

print("=" * 70)

Time Field Structuring and Validation


,metric,value
0,total_rows,621677
1,non_null_raw_time,621677
2,valid_hour_count,621677
3,valid_minute_count,621677
4,fully_valid_time_count,621677
5,invalid_time_count,0
6,distinct_hour_values,24
7,distinct_minute_values,60


----------------------------------------------------------------------
Sample preview:


,time_occurred_raw,time_occurred_str,occ_hour,occ_minute,is_valid_occ_time
0,2335,2335,23,35,True
1,1620,1620,16,20,True
2,1135,1135,11,35,True
3,230,0230,2,30,True
4,450,0450,4,50,True
5,2320,2320,23,20,True
6,545,0545,5,45,True
7,350,0350,3,50,True
8,2100,2100,21,0,True
9,950,0950,9,50,True


----------------------------------------------------------------------
Invalid time sample (if any):


,time_occurred_raw,time_occurred_str,occ_hour,occ_minute


----------------------------------------------------------------------
Distinct valid hours:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


## Step 6 — Create Time-of-Day Analytical Buckets

Now that the collision time field has been validated successfully, we create a higher-level time segmentation variable for easier analysis and reporting.

### Why this step matters
While the raw hour field is useful for detailed analysis, reporting and dashboards often benefit from broader business-readable time groups such as:

- early morning
- morning
- afternoon
- evening
- night

### Analytical value
This derived field can support:
- simplified dashboard filters
- executive summaries
- distribution analysis by time period
- comparison of collision concentration across parts of the day

### Methodological note
This is a derived analytical feature built from the already validated `occ_hour` field.
It does not replace the detailed hour variable; it complements it.

In [7]:
# ============================================================
# Cell 7: Create time-of-day analytical buckets
# ============================================================

def classify_time_of_day(hour):
    if pd.isna(hour):
        return np.nan
    hour = int(hour)

    if 0 <= hour <= 4:
        return "Late Night"
    elif 5 <= hour <= 8:
        return "Early Morning"
    elif 9 <= hour <= 11:
        return "Morning"
    elif 12 <= hour <= 16:
        return "Afternoon"
    elif 17 <= hour <= 20:
        return "Evening"
    elif 21 <= hour <= 23:
        return "Night"
    else:
        return "Unknown"

df["occ_time_of_day"] = df["occ_hour"].apply(classify_time_of_day)

# Summary
time_bucket_summary = (
    df["occ_time_of_day"]
    .value_counts(dropna=False)
    .rename_axis("occ_time_of_day")
    .reset_index(name="collision_count")
)

time_bucket_summary["share_percent"] = (
    time_bucket_summary["collision_count"] / len(df) * 100
).round(2)

print("=" * 70)
print("Time-of-Day Bucket Creation Completed")
print("=" * 70)
display(time_bucket_summary)

print("-" * 70)
print("Sample preview:")
display(
    df[
        [
            "time_occurred_raw",
            "time_occurred_str",
            "occ_hour",
            "occ_minute",
            "occ_time_of_day"
        ]
    ].head(15)
)

print("=" * 70)

Time-of-Day Bucket Creation Completed


,occ_time_of_day,collision_count,share_percent
0,Afternoon,181902,29.26
1,Evening,143910,23.15
2,Morning,81404,13.09
3,Early Morning,76881,12.37
4,Night,72685,11.69
5,Late Night,64895,10.44


----------------------------------------------------------------------
Sample preview:


,time_occurred_raw,time_occurred_str,occ_hour,occ_minute,occ_time_of_day
0,2335,2335,23,35,Night
1,1620,1620,16,20,Afternoon
2,1135,1135,11,35,Morning
3,230,0230,2,30,Late Night
4,450,0450,4,50,Late Night
5,2320,2320,23,20,Night
6,545,0545,5,45,Early Morning
7,350,0350,3,50,Late Night
8,2100,2100,21,0,Night
9,950,0950,9,50,Morning


## Step 7 — Standardize Key Text Fields

The raw dataset contains several text-based fields that are analytically useful but visually inconsistent due to extra spacing.

### Why this step matters
Text fields with irregular spacing can create:
- duplicate-looking categories
- grouping errors
- poor dashboard labels
- inconsistent filtering behavior

### Fields selected for cleaning
In this step, we standardize the spacing of the following fields:

- `area_name`
- `crime_code_description`
- `premise_description`
- `address`
- `cross_street`

### Cleaning rule applied
For each selected field, we:
- preserve missing values as missing
- remove leading and trailing spaces
- collapse repeated internal spaces into a single space

### Methodological note
This is a formatting-cleaning step, not a semantic transformation step.  
We do not change the meaning, casing logic, or category definitions at this stage.

In [8]:
# ============================================================
# Cell 8: Clean and standardize selected text fields
# ============================================================

text_columns_to_clean = [
    "area_name",
    "crime_code_description",
    "premise_description",
    "address",
    "cross_street"
]

def clean_text_spacing(value):
    if pd.isna(value):
        return value
    value = str(value)
    value = value.strip()
    value = " ".join(value.split())
    return value

# Preserve raw versions for audit
for col in text_columns_to_clean:
    df[f"{col}_raw"] = df[col]

# Apply cleaning
for col in text_columns_to_clean:
    df[col] = df[col].apply(clean_text_spacing)

# Build audit summary
text_cleaning_audit = []

for col in text_columns_to_clean:
    raw_col = f"{col}_raw"
    changed_count = (
        (df[raw_col].fillna("__MISSING__") != df[col].fillna("__MISSING__"))
    ).sum()
    
    text_cleaning_audit.append({
        "column": col,
        "non_null_count": df[col].notna().sum(),
        "changed_rows_count": int(changed_count),
        "missing_count": df[col].isna().sum(),
        "nunique_after_cleaning": df[col].nunique(dropna=True)
    })

text_cleaning_audit = pd.DataFrame(text_cleaning_audit)

print("=" * 70)
print("Selected Text Fields Cleaning Completed")
print("=" * 70)
display(text_cleaning_audit)

print("-" * 70)
print("Before/after sample preview:")
sample_preview = df[
    [
        "address_raw", "address",
        "cross_street_raw", "cross_street",
        "premise_description_raw", "premise_description"
    ]
].head(10)

display(sample_preview)

print("=" * 70)

Selected Text Fields Cleaning Completed


,column,non_null_count,changed_rows_count,missing_count,nunique_after_cleaning
0,area_name,621677,0,0,21
1,crime_code_description,621677,0,0,1
2,premise_description,620717,0,960,124
3,address,621677,505582,0,29442
4,cross_street,592217,467099,29460,21384


----------------------------------------------------------------------
Before/after sample preview:


,address_raw,address,cross_street_raw,cross_street,premise_description_raw,premise_description
0,WILTON PL,WILTON PL,6TH ST,6TH ST,STREET,STREET
1,NATIONAL BL,NATIONAL BL,MOTOR AV,MOTOR AV,STREET,STREET
2,PALMS BL,PALMS BL,ROSEWOOD AV,ROSEWOOD AV,STREET,STREET
3,IROLO ST,IROLO ST,SAN MARINO ST,SAN MARINO ST,STREET,STREET
4,JEFFERSON BL,JEFFERSON BL,NORMANDIE AV,NORMANDIE AV,STREET,STREET
5,JEFFERSON BL,JEFFERSON BL,W WESTERN,W WESTERN,STREET,STREET
6,N BROADWAY,N BROADWAY,W EASTLAKE AV,W EASTLAKE AV,STREET,STREET
7,1ST,1ST,CENTRAL,CENTRAL,STREET,STREET
8,MARTIN LUTHER KING JR,MARTIN LUTHER KING JR,ARLINGTON AV,ARLINGTON AV,STREET,STREET
9,MAIN,MAIN,JOHNSTON,JOHNSTON,STREET,STREET


## Step 8 — Assess and Clean the Victim Age Field

The `victim_age` field is analytically important, but age data often contains missing values, zeros, or implausible values that must be handled carefully.

### Why this step matters
Victim age is essential for:
- demographic analysis
- vulnerable user profiling
- age-band segmentation
- dashboard filtering
- injury and risk interpretation

### Cleaning approach
This step follows a conservative quality-control logic:

- preserve the original raw field
- profile the observed age distribution
- identify potentially invalid or implausible values
- create a cleaned analytical version of age
- avoid unjustified imputation

### Validation principle
We do not assume that every numeric age value is valid.  
Instead, we define a transparent rule to separate:
- usable ages
- missing ages
- invalid or implausible ages

### Working rule for this project
For analytical use, we will treat age as valid only when it falls within a reasonable human range.
At this stage, we will test and document the range before locking the final cleaned variable.

In [9]:
# ============================================================
# Cell 9: Profile and clean victim_age
# ============================================================

# Preserve raw age field
df["victim_age_raw"] = df["victim_age"]

# Basic profiling
age_profile_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "non_null_age_count",
        "missing_age_count",
        "min_age_raw",
        "max_age_raw",
        "mean_age_raw",
        "median_age_raw",
        "zero_age_count",
        "negative_age_count",
        "age_over_100_count"
    ],
    "value": [
        len(df),
        df["victim_age"].notna().sum(),
        df["victim_age"].isna().sum(),
        df["victim_age"].min(),
        df["victim_age"].max(),
        round(df["victim_age"].mean(), 2) if df["victim_age"].notna().any() else np.nan,
        df["victim_age"].median(),
        (df["victim_age"] == 0).sum(),
        (df["victim_age"] < 0).sum(),
        (df["victim_age"] > 100).sum()
    ]
})

# Define a transparent validity rule
# Keep ages between 1 and 100 inclusive as analytically valid for now
df["victim_age_clean"] = df["victim_age"].where(df["victim_age"].between(1, 100, inclusive="both"))

df["is_valid_victim_age"] = df["victim_age_clean"].notna()

age_cleaning_summary = pd.DataFrame({
    "metric": [
        "valid_age_count",
        "invalid_or_missing_age_count",
        "invalid_due_to_range_count",
        "min_valid_age",
        "max_valid_age",
        "mean_valid_age",
        "median_valid_age"
    ],
    "value": [
        df["is_valid_victim_age"].sum(),
        (~df["is_valid_victim_age"]).sum(),
        (
            df["victim_age"].notna() &
            ~df["victim_age"].between(1, 100, inclusive="both")
        ).sum(),
        df["victim_age_clean"].min(),
        df["victim_age_clean"].max(),
        round(df["victim_age_clean"].mean(), 2) if df["victim_age_clean"].notna().any() else np.nan,
        df["victim_age_clean"].median()
    ]
})

print("=" * 70)
print("Victim Age Profiling")
print("=" * 70)
display(age_profile_summary)

print("-" * 70)
print("Victim Age Cleaning Summary")
print("-" * 70)
display(age_cleaning_summary)

print("-" * 70)
print("Sample problematic age values:")
problem_age_sample = df.loc[
    df["victim_age"].notna() & ~df["victim_age"].between(1, 100, inclusive="both"),
    ["victim_age_raw", "victim_age", "victim_age_clean", "is_valid_victim_age"]
].head(20)

display(problem_age_sample)

print("-" * 70)
print("Top 20 raw age frequencies:")
display(
    df["victim_age"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("victim_age")
    .reset_index(name="count")
)

print("=" * 70)

Victim Age Profiling


,metric,value
0,total_rows,621677.0
1,non_null_age_count,533483.0
2,missing_age_count,88194.0
3,min_age_raw,10.0
4,max_age_raw,99.0
5,mean_age_raw,41.4
6,median_age_raw,39.0
7,zero_age_count,0.0
8,negative_age_count,0.0
9,age_over_100_count,0.0


----------------------------------------------------------------------
Victim Age Cleaning Summary
----------------------------------------------------------------------


,metric,value
0,valid_age_count,533483.0
1,invalid_or_missing_age_count,88194.0
2,invalid_due_to_range_count,0.0
3,min_valid_age,10.0
4,max_valid_age,99.0
5,mean_valid_age,41.4
6,median_valid_age,39.0


----------------------------------------------------------------------
Sample problematic age values:


,victim_age_raw,victim_age,victim_age_clean,is_valid_victim_age


----------------------------------------------------------------------
Top 20 raw age frequencies:


,victim_age,count
0,NaN,88194
1,30.0,18511
2,25.0,16593
3,40.0,15291
4,35.0,14681
5,27.0,14672
6,28.0,14514
7,23.0,14469
8,24.0,14465
9,50.0,14424


## Step 9 — Profile and Standardize Victim Sex and Victim Descent

The demographic categorical fields `victim_sex` and `victim_descent` are important for downstream demographic analysis, but they must first be standardized and documented.

### Why this step matters
These fields can support:
- demographic breakdowns
- vulnerability analysis
- dashboard filters
- subgroup comparisons

### Cleaning approach
This step focuses on structural consistency rather than category reinterpretation.

For each field, we will:
- preserve the raw version
- inspect missingness
- inspect unique values
- remove extra spaces
- standardize letter casing
- create cleaned analytical versions

### Methodological note
At this stage, we do not remap demographic codes into long textual labels.
We first preserve the source coding system and make it consistent for reliable analysis.

In [10]:
# ============================================================
# Cell 10: Profile and clean victim_sex and victim_descent
# ============================================================

demo_columns = ["victim_sex", "victim_descent"]

def clean_demo_code(value):
    if pd.isna(value):
        return value
    value = str(value).strip()
    value = " ".join(value.split())
    value = value.upper()
    return value

# Preserve raw fields
for col in demo_columns:
    df[f"{col}_raw"] = df[col]

# Apply cleaning
for col in demo_columns:
    df[f"{col}_clean"] = df[col].apply(clean_demo_code)

# Create validity flags (simple non-missing structural validity for now)
df["has_victim_sex"] = df["victim_sex_clean"].notna()
df["has_victim_descent"] = df["victim_descent_clean"].notna()

# Summary table
demo_summary_rows = []

for col in demo_columns:
    clean_col = f"{col}_clean"
    raw_col = f"{col}_raw"
    
    changed_count = (
        df[raw_col].fillna("__MISSING__") != df[clean_col].fillna("__MISSING__")
    ).sum()
    
    demo_summary_rows.append({
        "column": col,
        "non_null_count": df[clean_col].notna().sum(),
        "missing_count": df[clean_col].isna().sum(),
        "unique_values_after_cleaning": df[clean_col].nunique(dropna=True),
        "changed_rows_count": int(changed_count)
    })

demo_cleaning_summary = pd.DataFrame(demo_summary_rows)

print("=" * 70)
print("Victim Demographic Code Cleaning Summary")
print("=" * 70)
display(demo_cleaning_summary)

print("-" * 70)
print("Victim Sex value counts:")
display(
    df["victim_sex_clean"]
    .value_counts(dropna=False)
    .rename_axis("victim_sex_clean")
    .reset_index(name="count")
)

print("-" * 70)
print("Victim Descent value counts:")
display(
    df["victim_descent_clean"]
    .value_counts(dropna=False)
    .rename_axis("victim_descent_clean")
    .reset_index(name="count")
)

print("-" * 70)
print("Sample preview:")
display(
    df[
        [
            "victim_sex_raw", "victim_sex_clean",
            "victim_descent_raw", "victim_descent_clean"
        ]
    ].head(15)
)

print("=" * 70)

Victim Demographic Code Cleaning Summary


,column,non_null_count,missing_count,unique_values_after_cleaning,changed_rows_count
0,victim_sex,610980,10697,6,0
1,victim_descent,610029,11648,20,0


----------------------------------------------------------------------
Victim Sex value counts:


,victim_sex_clean,count
0,M,366612
1,F,226576
2,X,17594
3,NaN,10697
4,H,186
5,N,11
6,-,1


----------------------------------------------------------------------
Victim Descent value counts:


,victim_descent_clean,count
0,H,235060
1,W,140905
2,O,90589
3,B,82238
4,X,29918
5,A,22325
6,NaN,11648
7,K,4553
8,F,1776
9,C,918


----------------------------------------------------------------------
Sample preview:


,victim_sex_raw,victim_sex_clean,victim_descent_raw,victim_descent_clean
0,F,F,W,W
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,M,M,H,H
4,M,M,H,H
5,F,F,H,H
6,M,M,X,X
7,M,M,H,H
8,M,M,B,B
9,M,M,H,H


## Step 10 — Create Analytical Validation Rules for Victim Sex and Victim Descent

After structural cleaning, the next step is analytical validation.

### Why this step matters
Not every source code is equally suitable for immediate analytical use.
Some codes are standard and interpretable, while others are rare, placeholder-like, or operationally unclear.

### Validation strategy
We apply a conservative analytical rule:

#### For `victim_sex`
We keep only the clearly interpretable analytical values:
- `M`
- `F`
- `X`

All other values will be treated as missing in the strict analytical field.

#### For `victim_descent`
We do **not** discard valid source codes at this stage.
Instead:
- we preserve the cleaned source code
- flag placeholder-like values such as `-`
- keep the full coded field available for later mapping through documentation or lookup logic

### Methodological note
This step creates analysis-ready fields without overwriting the cleaned source-preserved versions.
That helps us maintain traceability between:
- raw values
- structurally cleaned values
- analytically validated values

In [11]:
# ============================================================
# Cell 11: Create analytical validation fields for victim demographics
# ============================================================

# ------------------------------------------------------------
# Victim sex: strict analytical version
# ------------------------------------------------------------
valid_victim_sex_values = {"M", "F", "X"}

df["victim_sex_analytical"] = df["victim_sex_clean"].where(
    df["victim_sex_clean"].isin(valid_victim_sex_values)
)

df["is_valid_victim_sex_analytical"] = df["victim_sex_analytical"].notna()

# ------------------------------------------------------------
# Victim descent: keep cleaned source code, but flag placeholder-like values
# ------------------------------------------------------------
placeholder_descent_values = {"-"}

df["victim_descent_analytical"] = df["victim_descent_clean"].where(
    ~df["victim_descent_clean"].isin(placeholder_descent_values)
)

df["is_valid_victim_descent_analytical"] = df["victim_descent_analytical"].notna()

# ------------------------------------------------------------
# Summary tables
# ------------------------------------------------------------
victim_sex_validation_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "non_null_clean_count",
        "valid_analytical_count",
        "invalid_or_unknown_count",
        "distinct_valid_analytical_values"
    ],
    "value": [
        len(df),
        df["victim_sex_clean"].notna().sum(),
        df["victim_sex_analytical"].notna().sum(),
        df["victim_sex_analytical"].isna().sum(),
        df["victim_sex_analytical"].nunique(dropna=True)
    ]
})

victim_descent_validation_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "non_null_clean_count",
        "valid_analytical_count",
        "invalid_placeholder_count",
        "missing_after_validation_count",
        "distinct_valid_analytical_values"
    ],
    "value": [
        len(df),
        df["victim_descent_clean"].notna().sum(),
        df["victim_descent_analytical"].notna().sum(),
        df["victim_descent_clean"].isin(placeholder_descent_values).sum(),
        df["victim_descent_analytical"].isna().sum(),
        df["victim_descent_analytical"].nunique(dropna=True)
    ]
})

print("=" * 70)
print("Victim Sex — Analytical Validation Summary")
print("=" * 70)
display(victim_sex_validation_summary)

print("-" * 70)
print("Victim Sex analytical value counts:")
display(
    df["victim_sex_analytical"]
    .value_counts(dropna=False)
    .rename_axis("victim_sex_analytical")
    .reset_index(name="count")
)

print("=" * 70)
print("Victim Descent — Analytical Validation Summary")
print("=" * 70)
display(victim_descent_validation_summary)

print("-" * 70)
print("Victim Descent analytical value counts:")
display(
    df["victim_descent_analytical"]
    .value_counts(dropna=False)
    .rename_axis("victim_descent_analytical")
    .reset_index(name="count")
)

print("-" * 70)
print("Rows with non-standard victim_sex codes:")
display(
    df.loc[
        df["victim_sex_clean"].notna() & ~df["victim_sex_clean"].isin(valid_victim_sex_values),
        ["victim_sex_raw", "victim_sex_clean", "victim_sex_analytical"]
    ].drop_duplicates().sort_values("victim_sex_clean")
)

print("=" * 70)

Victim Sex — Analytical Validation Summary


,metric,value
0,total_rows,621677
1,non_null_clean_count,610980
2,valid_analytical_count,610782
3,invalid_or_unknown_count,10895
4,distinct_valid_analytical_values,3


----------------------------------------------------------------------
Victim Sex analytical value counts:


,victim_sex_analytical,count
0,M,366612
1,F,226576
2,X,17594
3,NaN,10895


Victim Descent — Analytical Validation Summary


,metric,value
0,total_rows,621677
1,non_null_clean_count,610029
2,valid_analytical_count,610026
3,invalid_placeholder_count,3
4,missing_after_validation_count,11651
5,distinct_valid_analytical_values,19


----------------------------------------------------------------------
Victim Descent analytical value counts:


,victim_descent_analytical,count
0,H,235060
1,W,140905
2,O,90589
3,B,82238
4,X,29918
5,A,22325
6,NaN,11651
7,K,4553
8,F,1776
9,C,918


----------------------------------------------------------------------
Rows with non-standard victim_sex codes:


,victim_sex_raw,victim_sex_clean,victim_sex_analytical
133303,-,-,NaN
9295,H,H,NaN
538250,N,N,NaN


## Step 11 — Assess and Standardize Premise Code and Premise Description

The premise fields describe the type of location associated with each collision record.
These fields are important because they support spatial context analysis and will later feed the `dim_premise` table.

### Fields in scope
- `premise_code`
- `premise_description`

### Why this step matters
Premise information can be used for:
- location-type summaries
- dashboard breakdowns
- contextual risk analysis
- dimension table construction in SQL and BI models

### Cleaning objectives
In this step, we will:
- preserve the raw premise fields
- standardize the data type of `premise_code`
- inspect missingness
- evaluate one-to-one versus one-to-many relationships between code and description
- create cleaned analytical fields suitable for dimension-building

### Methodological note
This step focuses on structural readiness and consistency.
We do not yet build the final dimension table here; we first validate whether the premise fields are sufficiently clean and stable.

In [12]:
# ============================================================
# Cell 12: Profile and clean premise_code / premise_description
# ============================================================

# Preserve raw fields
df["premise_code_raw"] = df["premise_code"]
df["premise_description_raw"] = df["premise_description"]

# Standardize premise_code to nullable integer where possible
df["premise_code_clean"] = pd.to_numeric(df["premise_code"], errors="coerce").astype("Int64")

# premise_description was already spacing-cleaned earlier
df["premise_description_clean"] = df["premise_description"]

# Missingness and basic profiling
premise_profile_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "premise_code_non_null_count",
        "premise_code_missing_count",
        "premise_description_non_null_count",
        "premise_description_missing_count",
        "distinct_premise_codes",
        "distinct_premise_descriptions"
    ],
    "value": [
        len(df),
        df["premise_code_clean"].notna().sum(),
        df["premise_code_clean"].isna().sum(),
        df["premise_description_clean"].notna().sum(),
        df["premise_description_clean"].isna().sum(),
        df["premise_code_clean"].nunique(dropna=True),
        df["premise_description_clean"].nunique(dropna=True)
    ]
})

# Code-to-description consistency check
premise_mapping_check = (
    df.loc[df["premise_code_clean"].notna() & df["premise_description_clean"].notna(),
           ["premise_code_clean", "premise_description_clean"]]
    .drop_duplicates()
    .groupby("premise_code_clean")
    .agg(description_count=("premise_description_clean", "nunique"))
    .reset_index()
)

premise_codes_with_multiple_descriptions = premise_mapping_check.loc[
    premise_mapping_check["description_count"] > 1
].copy()

# Description-to-code consistency check
premise_reverse_mapping_check = (
    df.loc[df["premise_code_clean"].notna() & df["premise_description_clean"].notna(),
           ["premise_code_clean", "premise_description_clean"]]
    .drop_duplicates()
    .groupby("premise_description_clean")
    .agg(code_count=("premise_code_clean", "nunique"))
    .reset_index()
)

premise_descriptions_with_multiple_codes = premise_reverse_mapping_check.loc[
    premise_reverse_mapping_check["code_count"] > 1
].copy()

print("=" * 70)
print("Premise Fields Profiling Summary")
print("=" * 70)
display(premise_profile_summary)

print("-" * 70)
print("Premise code -> description consistency summary:")
print(f"Codes with more than one description: {len(premise_codes_with_multiple_descriptions)}")
display(premise_codes_with_multiple_descriptions.head(20))

print("-" * 70)
print("Premise description -> code consistency summary:")
print(f"Descriptions with more than one code: {len(premise_descriptions_with_multiple_codes)}")
display(premise_descriptions_with_multiple_codes.head(20))

print("-" * 70)
print("Top premise descriptions:")
display(
    df["premise_description_clean"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("premise_description_clean")
    .reset_index(name="count")
)

print("-" * 70)
print("Sample premise pairs:")
display(
    df[
        [
            "premise_code_raw",
            "premise_code_clean",
            "premise_description_raw",
            "premise_description_clean"
        ]
    ].head(15)
)

print("=" * 70)

Premise Fields Profiling Summary


,metric,value
0,total_rows,621677
1,premise_code_non_null_count,620718
2,premise_code_missing_count,959
3,premise_description_non_null_count,620717
4,premise_description_missing_count,960
5,distinct_premise_codes,125
6,distinct_premise_descriptions,124


----------------------------------------------------------------------
Premise code -> description consistency summary:
Codes with more than one description: 0


,premise_code_clean,description_count


----------------------------------------------------------------------
Premise description -> code consistency summary:
Descriptions with more than one code: 0


,premise_description_clean,code_count


----------------------------------------------------------------------
Top premise descriptions:


,premise_description_clean,count
0,STREET,591306
1,PARKING LOT,19487
2,SIDEWALK,4183
3,ALLEY,1208
4,DRIVEWAY,1127
5,NaN,960
6,FREEWAY,620
7,SINGLE FAMILY DWELLING,378
8,GAS STATION,367
9,TRANSPORTATION FACILITY (AIRPORT),221


----------------------------------------------------------------------
Sample premise pairs:


,premise_code_raw,premise_code_clean,premise_description_raw,premise_description_clean
0,101.0,101,STREET,STREET
1,101.0,101,STREET,STREET
2,101.0,101,STREET,STREET
3,101.0,101,STREET,STREET
4,101.0,101,STREET,STREET
5,101.0,101,STREET,STREET
6,101.0,101,STREET,STREET
7,101.0,101,STREET,STREET
8,101.0,101,STREET,STREET
9,101.0,101,STREET,STREET


## Step 12 — Build the Clean Premise Analytical Layer

The premise fields have passed the consistency checks successfully, so we can now convert them into a clean analytical layer suitable for dimension-building.

### Why this step matters
Since the mapping between `premise_code` and `premise_description` is stable, we can safely prepare a premise reference structure that will support:

- grouped reporting
- dashboard filtering
- data modeling
- dimension-table creation

### What we will do
In this step, we will:
- create final analytical premise fields
- isolate rows with missing premise information
- build a distinct premise reference table
- verify the uniqueness of the final premise dimension key

### Methodological note
This step does not yet export the final files.
It prepares the validated premise reference layer that will later be saved as part of the cleaned project outputs.

In [13]:
# ============================================================
# Cell 13: Build clean premise analytical layer and dim_premise
# ============================================================

# Final analytical premise fields
df["premise_code_analytical"] = df["premise_code_clean"]
df["premise_description_analytical"] = df["premise_description_clean"]

# Flag rows with incomplete premise information
df["has_complete_premise"] = (
    df["premise_code_analytical"].notna() &
    df["premise_description_analytical"].notna()
)

# Rows with incomplete premise information
premise_missing_rows = df.loc[
    ~df["has_complete_premise"],
    [
        "premise_code_raw",
        "premise_code_analytical",
        "premise_description_raw",
        "premise_description_analytical"
    ]
].copy()

# Build dim_premise candidate
dim_premise = (
    df.loc[
        df["has_complete_premise"],
        ["premise_code_analytical", "premise_description_analytical"]
    ]
    .drop_duplicates()
    .sort_values(["premise_code_analytical", "premise_description_analytical"])
    .reset_index(drop=True)
)

# Rename columns to final dimension-style names
dim_premise = dim_premise.rename(columns={
    "premise_code_analytical": "premise_code",
    "premise_description_analytical": "premise_description"
})

# Validation checks
dim_premise_validation = pd.DataFrame({
    "metric": [
        "rows_in_dim_premise",
        "distinct_premise_code_count",
        "distinct_premise_description_count",
        "duplicate_premise_code_rows",
        "duplicate_premise_description_rows",
        "rows_with_incomplete_premise_in_fact"
    ],
    "value": [
        len(dim_premise),
        dim_premise["premise_code"].nunique(dropna=True),
        dim_premise["premise_description"].nunique(dropna=True),
        dim_premise.duplicated(subset=["premise_code"]).sum(),
        dim_premise.duplicated(subset=["premise_description"]).sum(),
        (~df["has_complete_premise"]).sum()
    ]
})

print("=" * 70)
print("dim_premise Candidate Built Successfully")
print("=" * 70)
display(dim_premise_validation)

print("-" * 70)
print("dim_premise preview:")
display(dim_premise.head(20))

print("-" * 70)
print("Rows with incomplete premise data:")
display(premise_missing_rows.head(20))

print("=" * 70)

dim_premise Candidate Built Successfully


,metric,value
0,rows_in_dim_premise,124
1,distinct_premise_code_count,124
2,distinct_premise_description_count,124
3,duplicate_premise_code_rows,0
4,duplicate_premise_description_rows,0
5,rows_with_incomplete_premise_in_fact,960


----------------------------------------------------------------------
dim_premise preview:


,premise_code,premise_description
0,101,STREET
1,102,SIDEWALK
2,103,ALLEY
3,104,DRIVEWAY
4,105,PEDESTRIAN OVERCROSSING
5,106,TUNNEL
6,107,VACANT LOT
7,108,PARKING LOT
8,109,PARK/PLAYGROUND
9,110,FREEWAY


----------------------------------------------------------------------
Rows with incomplete premise data:


,premise_code_raw,premise_code_analytical,premise_description_raw,premise_description_analytical
1774,NaN,<NA>,NaN,NaN
1789,NaN,<NA>,NaN,NaN
1791,NaN,<NA>,NaN,NaN
1792,NaN,<NA>,NaN,NaN
1823,NaN,<NA>,NaN,NaN
1840,NaN,<NA>,NaN,NaN
1841,NaN,<NA>,NaN,NaN
1844,NaN,<NA>,NaN,NaN
1846,NaN,<NA>,NaN,NaN
1871,NaN,<NA>,NaN,NaN


## Step 13 — Parse the Location Field into Latitude and Longitude

The raw `location` field is currently stored as a text coordinate pair such as:

`(34.063, -118.3141)`

To make this field analytically usable, we need to extract its two numeric components:
- latitude
- longitude

### Why this step matters
Geospatial analysis depends on having structured numeric coordinates.
This supports:
- map visualizations
- hotspot summaries
- area-level geospatial context
- coordinate validation checks

### Transformation logic
In this step, we will:
- preserve the raw location field
- parse latitude and longitude from the raw text pattern
- convert both values to numeric form
- assess parsing success before applying geographic validity rules

### Methodological note
This step focuses only on structural extraction.
Geographic validity checks will be applied in the following step.

In [14]:
# ============================================================
# Cell 14: Parse location into latitude and longitude
# ============================================================

# Preserve raw location field
df["location_raw"] = df["location"]

# Extract latitude and longitude using regex
location_extracted = df["location"].astype(str).str.extract(
    r"\(\s*([\-]?\d+(?:\.\d+)?)\s*,\s*([\-]?\d+(?:\.\d+)?)\s*\)"
)

df["latitude"] = pd.to_numeric(location_extracted[0], errors="coerce")
df["longitude"] = pd.to_numeric(location_extracted[1], errors="coerce")

# Build parsing summary
location_parsing_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "non_null_location_raw_count",
        "parsed_latitude_count",
        "parsed_longitude_count",
        "rows_with_both_coordinates_parsed",
        "rows_with_failed_location_parsing"
    ],
    "value": [
        len(df),
        df["location_raw"].notna().sum(),
        df["latitude"].notna().sum(),
        df["longitude"].notna().sum(),
        (df["latitude"].notna() & df["longitude"].notna()).sum(),
        (~(df["latitude"].notna() & df["longitude"].notna())).sum()
    ]
})

print("=" * 70)
print("Location Parsing Summary")
print("=" * 70)
display(location_parsing_summary)

print("-" * 70)
print("Sample parsed location rows:")
display(
    df[
        [
            "location_raw",
            "latitude",
            "longitude"
        ]
    ].head(15)
)

print("-" * 70)
print("Sample failed parsing rows (if any):")
failed_location_sample = df.loc[
    ~(df["latitude"].notna() & df["longitude"].notna()),
    ["location_raw", "latitude", "longitude"]
].head(20)

display(failed_location_sample)

print("=" * 70)

Location Parsing Summary


,metric,value
0,total_rows,621677
1,non_null_location_raw_count,621677
2,parsed_latitude_count,621677
3,parsed_longitude_count,621677
4,rows_with_both_coordinates_parsed,621677
5,rows_with_failed_location_parsing,0


----------------------------------------------------------------------
Sample parsed location rows:


,location_raw,latitude,longitude
0,"(34.063, -118.3141)",34.0630,-118.3141
1,"(34.029, -118.4113)",34.0290,-118.4113
2,"(34.0052, -118.4478)",34.0052,-118.4478
3,"(34.0545, -118.3009)",34.0545,-118.3009
4,"(34.0255, -118.3002)",34.0255,-118.3002
5,"(34.0256, -118.3089)",34.0256,-118.3089
6,"(34.0738, -118.2078)",34.0738,-118.2078
7,"(34.0492, -118.2391)",34.0492,-118.2391
8,"(34.0108, -118.3182)",34.0108,-118.3182
9,"(34.066, -118.2102)",34.0660,-118.2102


----------------------------------------------------------------------
Sample failed parsing rows (if any):


,location_raw,latitude,longitude


## Step 14 — Validate Geographic Coordinates

After extracting latitude and longitude successfully, the next step is to verify whether the coordinates are geographically plausible.

### Why this step matters
Parsed coordinates may still be analytically unusable if they fall outside:
- valid global coordinate ranges
- the expected geographic region of the project

### Validation levels used
We apply two layers of validation:

#### 1) Global coordinate validity
Checks whether:
- latitude is between -90 and 90
- longitude is between -180 and 180

#### 2) Los Angeles analytical plausibility range
Since this project focuses on Los Angeles traffic collisions, we define a broad local bounding range for quality control:

- latitude between 33.6 and 34.4
- longitude between -118.7 and -117.9

### Analytical purpose
This allows us to distinguish between:
- structurally parsed coordinates
- globally valid coordinates
- locally plausible Los Angeles coordinates

### Methodological note
The local plausibility range is used as a practical analytical validation rule, not as a legal or cartographic boundary definition.

In [15]:
# ============================================================
# Cell 15: Validate coordinates globally and within LA plausibility range
# ============================================================

# ------------------------------------------------------------
# Global validity
# ------------------------------------------------------------
df["is_valid_lat_global"] = df["latitude"].between(-90, 90, inclusive="both")
df["is_valid_lon_global"] = df["longitude"].between(-180, 180, inclusive="both")
df["is_valid_coordinates_global"] = df["is_valid_lat_global"] & df["is_valid_lon_global"]

# ------------------------------------------------------------
# Los Angeles plausibility range (broad analytical range)
# ------------------------------------------------------------
LA_LAT_MIN, LA_LAT_MAX = 33.6, 34.4
LA_LON_MIN, LA_LON_MAX = -118.7, -117.9

df["is_valid_lat_la_range"] = df["latitude"].between(LA_LAT_MIN, LA_LAT_MAX, inclusive="both")
df["is_valid_lon_la_range"] = df["longitude"].between(LA_LON_MIN, LA_LON_MAX, inclusive="both")
df["has_valid_coordinates"] = df["is_valid_lat_la_range"] & df["is_valid_lon_la_range"]

# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------
coordinate_validation_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "globally_valid_coordinates_count",
        "globally_invalid_coordinates_count",
        "la_range_valid_coordinates_count",
        "la_range_invalid_coordinates_count",
        "min_latitude",
        "max_latitude",
        "min_longitude",
        "max_longitude"
    ],
    "value": [
        len(df),
        df["is_valid_coordinates_global"].sum(),
        (~df["is_valid_coordinates_global"]).sum(),
        df["has_valid_coordinates"].sum(),
        (~df["has_valid_coordinates"]).sum(),
        df["latitude"].min(),
        df["latitude"].max(),
        df["longitude"].min(),
        df["longitude"].max()
    ]
})

print("=" * 70)
print("Coordinate Validation Summary")
print("=" * 70)
display(coordinate_validation_summary)

print("-" * 70)
print("Rows outside Los Angeles plausibility range:")
outside_la_sample = df.loc[
    ~df["has_valid_coordinates"],
    ["location_raw", "latitude", "longitude"]
].head(20)

display(outside_la_sample)

print("-" * 70)
print("Sample valid coordinate rows:")
display(
    df.loc[
        df["has_valid_coordinates"],
        ["location_raw", "latitude", "longitude"]
    ].head(15)
)

print("=" * 70)

Coordinate Validation Summary


,metric,value
0,total_rows,621677.0000
1,globally_valid_coordinates_count,621677.0000
2,globally_invalid_coordinates_count,0.0000
3,la_range_valid_coordinates_count,620684.0000
4,la_range_invalid_coordinates_count,993.0000
5,min_latitude,0.0000
6,max_latitude,34.6920
7,min_longitude,-118.6673
8,max_longitude,0.0000


----------------------------------------------------------------------
Rows outside Los Angeles plausibility range:


,location_raw,latitude,longitude
85,"(0.0, 0.0)",0.0,0.0
2568,"(0.0, 0.0)",0.0,0.0
2571,"(0.0, 0.0)",0.0,0.0
2577,"(0.0, 0.0)",0.0,0.0
2664,"(0.0, 0.0)",0.0,0.0
2673,"(0.0, 0.0)",0.0,0.0
2674,"(0.0, 0.0)",0.0,0.0
2676,"(0.0, 0.0)",0.0,0.0
2679,"(0.0, 0.0)",0.0,0.0
2680,"(0.0, 0.0)",0.0,0.0


----------------------------------------------------------------------
Sample valid coordinate rows:


,location_raw,latitude,longitude
0,"(34.063, -118.3141)",34.0630,-118.3141
1,"(34.029, -118.4113)",34.0290,-118.4113
2,"(34.0052, -118.4478)",34.0052,-118.4478
3,"(34.0545, -118.3009)",34.0545,-118.3009
4,"(34.0255, -118.3002)",34.0255,-118.3002
5,"(34.0256, -118.3089)",34.0256,-118.3089
6,"(34.0738, -118.2078)",34.0738,-118.2078
7,"(34.0492, -118.2391)",34.0492,-118.2391
8,"(34.0108, -118.3182)",34.0108,-118.3182
9,"(34.066, -118.2102)",34.0660,-118.2102


## Step 15 — Build the Date Dimension from the Occurrence Date

The occurrence date has already been parsed and enriched with core temporal attributes.
We can now generate a clean date reference table for downstream analysis and modeling.

### Why this step matters
A date dimension helps standardize time-based analysis across:
- Python summaries
- SQL models
- Excel reporting
- Tableau dashboards
- Power BI dashboards

### Design choice
In this project, the primary analytical date dimension will be built from the collision occurrence date because it represents when the event actually happened.

### Fields included
The date dimension will include:
- occurrence date
- year
- month
- month name
- weekday number
- weekday name

### Methodological note
This table is created as a distinct reference layer and does not duplicate collision-level records.

In [16]:
# ============================================================
# Cell 16: Build dim_date from occurrence date
# ============================================================

dim_date = (
    df[
        [
            "date_occurred",
            "occ_year",
            "occ_month",
            "occ_month_name",
            "occ_weekday",
            "occ_weekday_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("date_occurred")
    .reset_index(drop=True)
    .rename(columns={
        "date_occurred": "date",
        "occ_year": "year",
        "occ_month": "month",
        "occ_month_name": "month_name",
        "occ_weekday": "weekday",
        "occ_weekday_name": "weekday_name"
    })
)

# Optional surrogate-style date key
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype("int64")

# Reorder columns
dim_date = dim_date[
    [
        "date_key",
        "date",
        "year",
        "month",
        "month_name",
        "weekday",
        "weekday_name"
    ]
]

dim_date_validation = pd.DataFrame({
    "metric": [
        "rows_in_dim_date",
        "distinct_date_key_count",
        "distinct_date_count",
        "min_date",
        "max_date",
        "duplicate_date_key_rows",
        "duplicate_date_rows"
    ],
    "value": [
        len(dim_date),
        dim_date["date_key"].nunique(),
        dim_date["date"].nunique(),
        dim_date["date"].min(),
        dim_date["date"].max(),
        dim_date.duplicated(subset=["date_key"]).sum(),
        dim_date.duplicated(subset=["date"]).sum()
    ]
})

print("=" * 70)
print("dim_date Built Successfully")
print("=" * 70)
display(dim_date_validation)

print("-" * 70)
print("dim_date preview:")
display(dim_date.head(20))

print("-" * 70)
print("dim_date tail preview:")
display(dim_date.tail(20))

print("=" * 70)

dim_date Built Successfully


,metric,value
0,rows_in_dim_date,5546
1,distinct_date_key_count,5546
2,distinct_date_count,5546
3,min_date,2010-01-01 00:00:00
4,max_date,2025-03-08 00:00:00
5,duplicate_date_key_rows,0
6,duplicate_date_rows,0


----------------------------------------------------------------------
dim_date preview:


,date_key,date,year,month,month_name,weekday,weekday_name
0,20100101,2010-01-01,2010,1,January,4,Friday
1,20100102,2010-01-02,2010,1,January,5,Saturday
2,20100103,2010-01-03,2010,1,January,6,Sunday
3,20100104,2010-01-04,2010,1,January,0,Monday
4,20100105,2010-01-05,2010,1,January,1,Tuesday
5,20100106,2010-01-06,2010,1,January,2,Wednesday
6,20100107,2010-01-07,2010,1,January,3,Thursday
7,20100108,2010-01-08,2010,1,January,4,Friday
8,20100109,2010-01-09,2010,1,January,5,Saturday
9,20100110,2010-01-10,2010,1,January,6,Sunday


----------------------------------------------------------------------
dim_date tail preview:


,date_key,date,year,month,month_name,weekday,weekday_name
5526,20250217,2025-02-17,2025,2,February,0,Monday
5527,20250218,2025-02-18,2025,2,February,1,Tuesday
5528,20250219,2025-02-19,2025,2,February,2,Wednesday
5529,20250220,2025-02-20,2025,2,February,3,Thursday
5530,20250221,2025-02-21,2025,2,February,4,Friday
5531,20250222,2025-02-22,2025,2,February,5,Saturday
5532,20250223,2025-02-23,2025,2,February,6,Sunday
5533,20250224,2025-02-24,2025,2,February,0,Monday
5534,20250225,2025-02-25,2025,2,February,1,Tuesday
5535,20250226,2025-02-26,2025,2,February,2,Wednesday


## Step 16 — Define the Final Clean Collision Base Schema

At this point, the core cleaning and standardization work has been completed for the main collision table.

### Why this step matters
During cleaning, we created many helper columns for:
- raw preservation
- validation
- auditing
- intermediate transformation checks

These are useful for notebook logic, but they should not all be carried into the final clean analytical base.

### Objective of this step
We now define the final schema of `collisions_clean` by selecting the columns that should remain in the cleaned base table.

### Selection logic
The final cleaned table should retain:
- source identifiers
- cleaned event dates and time structure
- core location and district fields
- cleaned demographic analytical fields
- validated premise fields
- geospatial coordinates and validity flags
- MO Codes raw field for later engineering in Notebook 03

### Methodological note
This step creates the clean analytical base table structure.
It does not yet save the file; we first validate the final selected schema.

In [17]:
# ============================================================
# Cell 17: Define final collisions_clean schema
# ============================================================

collisions_clean_columns = [
    # Primary identifiers
    "dr_number",

    # Core dates
    "date_reported",
    "date_occurred",

    # Time structure
    "time_occurred",
    "time_occurred_str",
    "occ_hour",
    "occ_minute",
    "occ_time_of_day",

    # Derived temporal fields
    "occ_year",
    "occ_month",
    "occ_month_name",
    "occ_weekday",
    "occ_weekday_name",

    # Area and reporting context
    "area_id",
    "area_name",
    "reporting_district",

    # Crime reference
    "crime_code",
    "crime_code_description",

    # MO codes kept for next notebook
    "mo_codes",

    # Demographics
    "victim_age_clean",
    "is_valid_victim_age",
    "victim_sex_analytical",
    "is_valid_victim_sex_analytical",
    "victim_descent_analytical",
    "is_valid_victim_descent_analytical",

    # Premise
    "premise_code_analytical",
    "premise_description_analytical",
    "has_complete_premise",

    # Address fields
    "address",
    "cross_street",

    # Raw location text + parsed coordinates
    "location",
    "latitude",
    "longitude",
    "has_valid_coordinates"
]

# Build final clean collision table candidate
collisions_clean = df[collisions_clean_columns].copy()

# Rename some analytical columns to cleaner final names
collisions_clean = collisions_clean.rename(columns={
    "victim_age_clean": "victim_age",
    "victim_sex_analytical": "victim_sex",
    "victim_descent_analytical": "victim_descent",
    "premise_code_analytical": "premise_code",
    "premise_description_analytical": "premise_description"
})

# Validation summary
collisions_clean_validation = pd.DataFrame({
    "metric": [
        "row_count",
        "column_count",
        "duplicate_dr_number_count",
        "non_null_date_occurred_count",
        "non_null_occ_hour_count",
        "non_null_area_name_count",
        "non_null_mo_codes_count",
        "non_null_victim_age_count",
        "non_null_victim_sex_count",
        "non_null_victim_descent_count",
        "non_null_premise_code_count",
        "valid_coordinate_count"
    ],
    "value": [
        len(collisions_clean),
        collisions_clean.shape[1],
        collisions_clean.duplicated(subset=["dr_number"]).sum(),
        collisions_clean["date_occurred"].notna().sum(),
        collisions_clean["occ_hour"].notna().sum(),
        collisions_clean["area_name"].notna().sum(),
        collisions_clean["mo_codes"].notna().sum(),
        collisions_clean["victim_age"].notna().sum(),
        collisions_clean["victim_sex"].notna().sum(),
        collisions_clean["victim_descent"].notna().sum(),
        collisions_clean["premise_code"].notna().sum(),
        collisions_clean["has_valid_coordinates"].sum()
    ]
})

print("=" * 70)
print("collisions_clean Candidate Schema Validation")
print("=" * 70)
display(collisions_clean_validation)

print("-" * 70)
print("Final selected columns:")
for i, col in enumerate(collisions_clean.columns, start=1):
    print(f"{i:02d}. {col}")

print("-" * 70)
print("Preview of collisions_clean:")
display(collisions_clean.head(10))

print("=" * 70)

collisions_clean Candidate Schema Validation


,metric,value
0,row_count,621677
1,column_count,34
2,duplicate_dr_number_count,0
3,non_null_date_occurred_count,621677
4,non_null_occ_hour_count,621677
5,non_null_area_name_count,621677
6,non_null_mo_codes_count,534353
7,non_null_victim_age_count,533483
8,non_null_victim_sex_count,610782
9,non_null_victim_descent_count,610026


----------------------------------------------------------------------
Final selected columns:
01. dr_number
02. date_reported
03. date_occurred
04. time_occurred
05. time_occurred_str
06. occ_hour
07. occ_minute
08. occ_time_of_day
09. occ_year
10. occ_month
11. occ_month_name
12. occ_weekday
13. occ_weekday_name
14. area_id
15. area_name
16. reporting_district
17. crime_code
18. crime_code_description
19. mo_codes
20. victim_age
21. is_valid_victim_age
22. victim_sex
23. is_valid_victim_sex_analytical
24. victim_descent
25. is_valid_victim_descent_analytical
26. premise_code
27. premise_description
28. has_complete_premise
29. address
30. cross_street
31. location
32. latitude
33. longitude
34. has_valid_coordinates
----------------------------------------------------------------------
Preview of collisions_clean:


,dr_number,date_reported,date_occurred,time_occurred,time_occurred_str,occ_hour,occ_minute,occ_time_of_day,occ_year,occ_month,occ_month_name,occ_weekday,occ_weekday_name,area_id,area_name,reporting_district,crime_code,crime_code_description,mo_codes,victim_age,is_valid_victim_age,victim_sex,is_valid_victim_sex_analytical,victim_descent,is_valid_victim_descent_analytical,premise_code,premise_description,has_complete_premise,address,cross_street,location,latitude,longitude,has_valid_coordinates
0,212013850,2021-09-03,2021-09-02,2335,2335,23,35,Night,2021,9,September,3,Thursday,20,Olympic,2021,997,TRAFFIC COLLISION,3004 3027 3034 4027 3036 3101 3401 3701,25.0,True,F,True,W,True,101,STREET,True,WILTON PL,6TH ST,"(34.063, -118.3141)",34.0630,-118.3141,True
1,221417787,2022-10-17,2022-10-17,1620,1620,16,20,Afternoon,2022,10,October,0,Monday,14,Pacific,1406,997,TRAFFIC COLLISION,4027 3011 3028 3034 3037 3101 3401 3701,21.0,True,NaN,False,NaN,False,101,STREET,True,NATIONAL BL,MOTOR AV,"(34.029, -118.4113)",34.0290,-118.4113,True
2,221418141,2022-10-26,2022-10-26,1135,1135,11,35,Morning,2022,10,October,2,Wednesday,14,Pacific,1434,997,TRAFFIC COLLISION,4027 3011 3025 3034 3037 3101 3401 3701,21.0,True,NaN,False,NaN,False,101,STREET,True,PALMS BL,ROSEWOOD AV,"(34.0052, -118.4478)",34.0052,-118.4478,True
3,222017859,2022-12-01,2022-12-01,230,0230,2,30,Late Night,2022,12,December,3,Thursday,20,Olympic,2044,997,TRAFFIC COLLISION,3003 0913 3026 3035 3037 3101 3401 3701 4020,33.0,True,M,True,H,True,101,STREET,True,IROLO ST,SAN MARINO ST,"(34.0545, -118.3009)",34.0545,-118.3009,True
4,190319651,2019-08-24,2019-08-24,450,0450,4,50,Late Night,2019,8,August,5,Saturday,3,Southwest,356,997,TRAFFIC COLLISION,3036 3004 3026 3101 4003,22.0,True,M,True,H,True,101,STREET,True,JEFFERSON BL,NORMANDIE AV,"(34.0255, -118.3002)",34.0255,-118.3002,True
5,190319680,2019-08-30,2019-08-30,2320,2320,23,20,Night,2019,8,August,4,Friday,3,Southwest,355,997,TRAFFIC COLLISION,3037 3006 3028 3030 3039 3101 4003,30.0,True,F,True,H,True,101,STREET,True,JEFFERSON BL,W WESTERN,"(34.0256, -118.3089)",34.0256,-118.3089,True
6,190413769,2019-08-25,2019-08-25,545,0545,5,45,Early Morning,2019,8,August,6,Sunday,4,Hollenbeck,422,997,TRAFFIC COLLISION,3101 3401 3701 3006 3030,NaN,False,M,True,X,True,101,STREET,True,N BROADWAY,W EASTLAKE AV,"(34.0738, -118.2078)",34.0738,-118.2078,True
7,190127578,2019-11-20,2019-11-20,350,0350,3,50,Late Night,2019,11,November,2,Wednesday,1,Central,128,997,TRAFFIC COLLISION,0605 3101 3401 3701 3011 3034,21.0,True,M,True,H,True,101,STREET,True,1ST,CENTRAL,"(34.0492, -118.2391)",34.0492,-118.2391,True
8,190319695,2019-08-30,2019-08-30,2100,2100,21,0,Night,2019,8,August,4,Friday,3,Southwest,374,997,TRAFFIC COLLISION,0605 4025 3037 3004 3025 3101,49.0,True,M,True,B,True,101,STREET,True,MARTIN LUTHER KING JR,ARLINGTON AV,"(34.0108, -118.3182)",34.0108,-118.3182,True
9,190411883,2019-07-06,2019-07-06,950,0950,9,50,Morning,2019,7,July,5,Saturday,4,Hollenbeck,423,997,TRAFFIC COLLISION,3101 3401 3701 3003 3025 3029,60.0,True,M,True,H,True,101,STREET,True,MAIN,JOHNSTON,"(34.066, -118.2102)",34.0660,-118.2102,True


## Step 17 — Build the Area Dimension

The area fields provide an important geographic and administrative grouping layer for the project.

### Fields in scope
- `area_id`
- `area_name`

### Why this step matters
A clean area dimension supports:
- area-based trend analysis
- district and geographic summaries
- SQL dimension modeling
- dashboard filters and slicers

### Validation goal
Before using these fields as a dimension, we need to confirm:
- whether each `area_id` maps to exactly one `area_name`
- whether each `area_name` maps to exactly one `area_id`
- whether any rows have missing area information

### Methodological note
As with the other dimension candidates, we first validate the mapping before treating it as a final reusable reference table.

In [18]:
# ============================================================
# Cell 18: Build and validate dim_area
# ============================================================

# Build raw candidate pairs
dim_area_candidate = (
    collisions_clean[["area_id", "area_name"]]
    .drop_duplicates()
    .sort_values(["area_id", "area_name"])
    .reset_index(drop=True)
)

# Validation checks
area_id_to_name_check = (
    dim_area_candidate
    .groupby("area_id")
    .agg(area_name_count=("area_name", "nunique"))
    .reset_index()
)

area_name_to_id_check = (
    dim_area_candidate
    .groupby("area_name")
    .agg(area_id_count=("area_id", "nunique"))
    .reset_index()
)

area_ids_with_multiple_names = area_id_to_name_check.loc[
    area_id_to_name_check["area_name_count"] > 1
].copy()

area_names_with_multiple_ids = area_name_to_id_check.loc[
    area_name_to_id_check["area_id_count"] > 1
].copy()

# Final dim_area
dim_area = dim_area_candidate.copy()

dim_area_validation = pd.DataFrame({
    "metric": [
        "rows_in_dim_area",
        "distinct_area_id_count",
        "distinct_area_name_count",
        "missing_area_id_rows",
        "missing_area_name_rows",
        "area_ids_with_multiple_names",
        "area_names_with_multiple_ids",
        "duplicate_area_id_rows",
        "duplicate_area_name_rows"
    ],
    "value": [
        len(dim_area),
        dim_area["area_id"].nunique(dropna=True),
        dim_area["area_name"].nunique(dropna=True),
        dim_area["area_id"].isna().sum(),
        dim_area["area_name"].isna().sum(),
        len(area_ids_with_multiple_names),
        len(area_names_with_multiple_ids),
        dim_area.duplicated(subset=["area_id"]).sum(),
        dim_area.duplicated(subset=["area_name"]).sum()
    ]
})

print("=" * 70)
print("dim_area Built Successfully")
print("=" * 70)
display(dim_area_validation)

print("-" * 70)
print("dim_area preview:")
display(dim_area)

print("-" * 70)
print("Area IDs with multiple names (if any):")
display(area_ids_with_multiple_names)

print("-" * 70)
print("Area names with multiple IDs (if any):")
display(area_names_with_multiple_ids)

print("=" * 70)

dim_area Built Successfully


,metric,value
0,rows_in_dim_area,21
1,distinct_area_id_count,21
2,distinct_area_name_count,21
3,missing_area_id_rows,0
4,missing_area_name_rows,0
5,area_ids_with_multiple_names,0
6,area_names_with_multiple_ids,0
7,duplicate_area_id_rows,0
8,duplicate_area_name_rows,0


----------------------------------------------------------------------
dim_area preview:


,area_id,area_name
0,1,Central
1,2,Rampart
2,3,Southwest
3,4,Hollenbeck
4,5,Harbor
5,6,Hollywood
6,7,Wilshire
7,8,West LA
8,9,Van Nuys
9,10,West Valley


----------------------------------------------------------------------
Area IDs with multiple names (if any):


,area_id,area_name_count


----------------------------------------------------------------------
Area names with multiple IDs (if any):


,area_name,area_id_count


## Step 18 — Save the Core Clean Outputs

After completing the main cleaning and validation steps, we now save the core outputs produced by Notebook 02.

### Outputs to be saved
This notebook is expected to produce the following cleaned base files:

- `collisions_clean.csv`
- `dim_date.csv`
- `dim_area.csv`
- `dim_premise.csv`
- `cleaning_summary_report.csv`

### Why this step matters
Saving the outputs at this stage provides:
- a stable analytical base for Notebook 03
- reproducible intermediate deliverables
- clean handoff files for SQL and BI preparation later
- a documented checkpoint in the project workflow

### Methodological note
The saved outputs should reflect the validated analytical versions of the tables, not the temporary helper fields used internally during notebook development.

In [19]:
# ============================================================
# Cell 19: Save cleaned outputs and build cleaning summary report
# ============================================================

# ------------------------------------------------------------
# Build cleaning summary report
# ------------------------------------------------------------
cleaning_summary_report = pd.DataFrame([
    {"component": "collisions_clean", "metric": "row_count", "value": len(collisions_clean)},
    {"component": "collisions_clean", "metric": "column_count", "value": collisions_clean.shape[1]},
    {"component": "collisions_clean", "metric": "duplicate_dr_number_count", "value": collisions_clean.duplicated(subset=["dr_number"]).sum()},
    {"component": "collisions_clean", "metric": "non_null_mo_codes_count", "value": collisions_clean["mo_codes"].notna().sum()},
    {"component": "collisions_clean", "metric": "non_null_victim_age_count", "value": collisions_clean["victim_age"].notna().sum()},
    {"component": "collisions_clean", "metric": "non_null_victim_sex_count", "value": collisions_clean["victim_sex"].notna().sum()},
    {"component": "collisions_clean", "metric": "non_null_victim_descent_count", "value": collisions_clean["victim_descent"].notna().sum()},
    {"component": "collisions_clean", "metric": "non_null_premise_code_count", "value": collisions_clean["premise_code"].notna().sum()},
    {"component": "collisions_clean", "metric": "valid_coordinate_count", "value": collisions_clean["has_valid_coordinates"].sum()},
    {"component": "dim_date", "metric": "row_count", "value": len(dim_date)},
    {"component": "dim_date", "metric": "min_date", "value": dim_date["date"].min()},
    {"component": "dim_date", "metric": "max_date", "value": dim_date["date"].max()},
    {"component": "dim_area", "metric": "row_count", "value": len(dim_area)},
    {"component": "dim_premise", "metric": "row_count", "value": len(dim_premise)},
    {"component": "dim_premise", "metric": "incomplete_premise_rows_in_fact", "value": (~df["has_complete_premise"]).sum()},
])

# ------------------------------------------------------------
# Define output file paths
# ------------------------------------------------------------
collisions_clean_path = CLEANED_DIR / "collisions_clean.csv"
dim_date_path = CLEANED_DIR / "dim_date.csv"
dim_area_path = CLEANED_DIR / "dim_area.csv"
dim_premise_path = CLEANED_DIR / "dim_premise.csv"
cleaning_summary_report_path = CLEANED_DIR / "cleaning_summary_report.csv"

# ------------------------------------------------------------
# Save files
# ------------------------------------------------------------
collisions_clean.to_csv(collisions_clean_path, index=False)
dim_date.to_csv(dim_date_path, index=False)
dim_area.to_csv(dim_area_path, index=False)
dim_premise.to_csv(dim_premise_path, index=False)
cleaning_summary_report.to_csv(cleaning_summary_report_path, index=False)

print("=" * 70)
print("Notebook 02 Outputs Saved Successfully")
print("=" * 70)
print(f"collisions_clean.csv         -> {collisions_clean_path}")
print(f"dim_date.csv                 -> {dim_date_path}")
print(f"dim_area.csv                 -> {dim_area_path}")
print(f"dim_premise.csv              -> {dim_premise_path}")
print(f"cleaning_summary_report.csv  -> {cleaning_summary_report_path}")

print("-" * 70)
print("Cleaning summary report preview:")
display(cleaning_summary_report)

print("-" * 70)
print("Saved file existence check:")
print(f"collisions_clean exists?        {collisions_clean_path.exists()}")
print(f"dim_date exists?                {dim_date_path.exists()}")
print(f"dim_area exists?                {dim_area_path.exists()}")
print(f"dim_premise exists?             {dim_premise_path.exists()}")
print(f"cleaning_summary_report exists? {cleaning_summary_report_path.exists()}")

print("=" * 70)

Notebook 02 Outputs Saved Successfully
collisions_clean.csv         -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\collisions_clean.csv
dim_date.csv                 -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\dim_date.csv
dim_area.csv                 -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\dim_area.csv
dim_premise.csv              -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\dim_premise.csv
cleaning_summary_report.csv  -> c:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\data\cleaned\cleaning_summary_report.csv
----------------------------------------------------------------------
Cleaning summary report preview:


,component,metric,value
0,collisions_clean,row_count,621677
1,collisions_clean,column_count,34
2,collisions_clean,duplicate_dr_number_count,0
3,collisions_clean,non_null_mo_codes_count,534353
4,collisions_clean,non_null_victim_age_count,533483
5,collisions_clean,non_null_victim_sex_count,610782
6,collisions_clean,non_null_victim_descent_count,610026
7,collisions_clean,non_null_premise_code_count,620718
8,collisions_clean,valid_coordinate_count,620684
9,dim_date,row_count,5546


----------------------------------------------------------------------
Saved file existence check:
collisions_clean exists?        True
dim_date exists?                True
dim_area exists?                True
dim_premise exists?             True
cleaning_summary_report exists? True


# Notebook 02 Conclusion

This notebook successfully transformed the raw Los Angeles traffic collision dataset into a clean, standardized, and analysis-ready analytical base.

## What was completed

### 1) Structural standardization
- raw source column names were converted into a consistent analytical naming format
- the working schema was stabilized for downstream Python, SQL, and BI usage

### 2) Date cleaning and temporal enrichment
- `date_reported` and `date_occurred` were parsed successfully with zero conversion failures
- core temporal features were derived from `date_occurred`, including:
  - year
  - month
  - month name
  - weekday
  - weekday name

### 3) Time field structuring
- `time_occurred` was standardized into a zero-padded HHMM format
- hour and minute fields were extracted successfully
- all rows passed time validity checks
- an analytical `occ_time_of_day` bucket was created for reporting and dashboard use

### 4) Text cleaning
- key text fields were standardized by removing extra spaces and normalizing formatting
- the largest formatting improvements were achieved in:
  - `address`
  - `cross_street`

### 5) Demographic field preparation
- `victim_age` was profiled and confirmed to contain no invalid numeric outliers in the observed non-null values
- analytical versions of:
  - `victim_sex`
  - `victim_descent`
  were prepared using transparent validation rules suitable for analysis

### 6) Premise layer validation
- `premise_code` and `premise_description` were standardized and validated
- the mapping between premise code and premise description was confirmed to be one-to-one
- a clean `dim_premise` reference table was successfully built

### 7) Geographic parsing and validation
- the `location` field was parsed successfully into:
  - `latitude`
  - `longitude`
- all rows were structurally parsed without regex failures
- geographic validation identified a limited number of locally invalid coordinates, mainly placeholder-style values such as `(0.0, 0.0)`
- a final `has_valid_coordinates` flag was created for geospatial use

### 8) Dimension table creation
The following reference tables were built successfully:
- `dim_date`
- `dim_area`
- `dim_premise`

### 9) Final clean base table creation
A final cleaned base table was assembled as:
- `collisions_clean.csv`

This table now contains the validated and analysis-ready columns required for:
- exploratory analysis
- MO code engineering
- SQL preparation
- Excel exports
- Tableau preparation
- Power BI preparation

## Key output figures

- raw / cleaned collision rows: **621,677**
- final columns in `collisions_clean`: **34**
- duplicate `dr_number` values: **0**
- non-null `mo_codes`: **534,353**
- non-null `victim_age`: **533,483**
- non-null analytical `victim_sex`: **610,782**
- non-null analytical `victim_descent`: **610,026**
- non-null `premise_code`: **620,718**
- valid geospatial coordinates in LA plausibility range: **620,684**
- rows in `dim_date`: **5,546**
- rows in `dim_area`: **21**
- rows in `dim_premise`: **124**
- incomplete premise rows in the fact table: **960**

## Files produced
- `collisions_clean.csv`
- `dim_date.csv`
- `dim_area.csv`
- `dim_premise.csv`
- `cleaning_summary_report.csv`

## Methodological result
This notebook established the project’s trusted cleaned analytical base.

The dataset is now ready for the next step:

**Notebook 03 — MO Codes Engineering**

That notebook will focus on:
- splitting multi-code MO fields
- building the collision-to-code bridge table
- joining the classified MO reference
- identifying unmapped codes
- creating a structured analytical MO layer for downstream reporting and modeling